In [2]:
# Installing required packages in Google Colab
!pip -q install yfinance lxml html5lib

# Here are the necessary libraries/modules
import requests
import pandas as pd
import numpy as np
import yfinance as yf
from time import sleep

# STEP 1: Read current S&P 500 constituents safely
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

sp500_table = pd.read_html(response.text)[0]

tickers = (
    sp500_table["Symbol"]
    .astype(str)
    .str.replace(".", "-", regex=False)
    .tolist()
)

print("Total tickers fetched:", len(tickers))


# STEP 2: Get market caps and rank stocks
market_caps = []

for i, ticker in enumerate(tickers, start=1):
    market_cap = None
    try:
        tk = yf.Ticker(ticker)

        # Try fast_info first
        try:
            fi = tk.fast_info
            market_cap = fi.get("market_cap", None)
        except Exception:
            pass

        # Fallback to info
        if market_cap is None:
            try:
                info = tk.info
                market_cap = info.get("marketCap", None)
            except Exception:
                pass

    except Exception:
        pass

    market_caps.append((ticker, market_cap))
    print(f"{i:>3}. {ticker}: {market_cap}")
    sleep(0.10)  # small pause to reduce request issues

market_cap_df = pd.DataFrame(market_caps, columns=["Ticker", "MarketCap"])
market_cap_df = market_cap_df.dropna(subset=["MarketCap"]).copy()
market_cap_df["MarketCap"] = pd.to_numeric(market_cap_df["MarketCap"], errors="coerce")
market_cap_df = market_cap_df.dropna(subset=["MarketCap"]).sort_values("MarketCap", ascending=False).reset_index(drop=True)

if len(market_cap_df) < 50:
    raise ValueError("Fewer than 50 valid tickers with market cap found. Please rerun later.")

# STEP 3: Freeze top 50 universe and add SPY benchmark
top50 = market_cap_df.head(50)["Ticker"].tolist()
download_tickers = top50 + ["SPY"]

print("\nTop 50 universe selected:")
print(top50)

# STEP 4: Download daily OHLCV data
raw_data = yf.download(
    tickers=download_tickers,
    start="2018-01-01",
    end="2024-01-01",
    auto_adjust=False,
    progress=False,
    threads=True
)

if raw_data.empty:
    raise ValueError("No price data downloaded. Please rerun.")


# STEP 5: Convert wide multi-index data into long format
# Note that yfinance usually returns columns like (PriceField, Ticker)
# I will stack the ticker level into rows
long_data = raw_data.stack(level=1).reset_index()

# Renaming safely based on actual structure
# Expected columns after stack: Date + ticker level + price fields
long_data = long_data.rename(columns={
    "level_0": "Date",
    "level_1": "Ticker",
    "Adj Close": "Adj Close",
    "Close": "Close",
    "High": "High",
    "Low": "Low",
    "Open": "Open",
    "Volume": "Volume"
})

# Some pandas versions may name the first two columns differently
cols = list(long_data.columns)
cols[0] = "Date"
cols[1] = "Ticker"
long_data.columns = cols

required_price_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
missing_cols = [c for c in required_price_cols if c not in long_data.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns after reshaping: {missing_cols}")

long_data = long_data[["Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]].copy()
long_data["Date"] = pd.to_datetime(long_data["Date"])
long_data = long_data.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Ensuring numeric types
for col in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]:
    long_data[col] = pd.to_numeric(long_data[col], errors="coerce")

# Dropping rows with missing core price info
long_data = long_data.dropna(subset=["Adj Close", "Volume"]).reset_index(drop=True)


# STEP 6: Feature engineering
# Daily return
long_data["Return"] = long_data.groupby("Ticker")["Adj Close"].pct_change()

# Next-day return target
long_data["NextDayReturn"] = long_data.groupby("Ticker")["Return"].shift(-1)

# 20-day momentum
long_data["Momentum_20D"] = long_data.groupby("Ticker")["Adj Close"].pct_change(20)

# 20-day rolling volatility of daily returns
long_data["Volatility_20D"] = (
    long_data.groupby("Ticker")["Return"]
    .rolling(window=20, min_periods=20)
    .std()
    .reset_index(level=0, drop=True)
)

# 20-day average volume
long_data["AvgVolume_20D"] = (
    long_data.groupby("Ticker")["Volume"]
    .rolling(window=20, min_periods=20)
    .mean()
    .reset_index(level=0, drop=True)
)

# Volume ratio
long_data["VolumeRatio_20D"] = long_data["Volume"] / long_data["AvgVolume_20D"]


# STEP 7: Add market return from SPY
spy_returns = (
    long_data[long_data["Ticker"] == "SPY"][["Date", "Return"]]
    .rename(columns={"Return": "MarketReturn"})
    .copy()
)

final_data = long_data.merge(spy_returns, on="Date", how="left")

# STEP 8: Cross-sectional z-scores by date
# IMPORTANT NOTE: computing using stock universe only, excluding SPY
stock_only = final_data[final_data["Ticker"] != "SPY"].copy()

signal_cols = ["Momentum_20D", "Volatility_20D", "VolumeRatio_20D"]

for col in signal_cols:
    daily_mean = stock_only.groupby("Date")[col].transform("mean")
    daily_std = stock_only.groupby("Date")[col].transform("std")
    stock_only[f"{col}_Z"] = (stock_only[col] - daily_mean) / daily_std


# STEP 9: Final modeling dataset
model_data = stock_only.dropna(subset=[
    "NextDayReturn",
    "Momentum_20D",
    "Volatility_20D",
    "VolumeRatio_20D",
    "MarketReturn",
    "Momentum_20D_Z",
    "Volatility_20D_Z",
    "VolumeRatio_20D_Z"
]).copy()

# Well, it's optional: remove infinities if any
model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

# Keeping only useful final columns
final_regression_dataset = model_data[
    [
        "Date",
        "Ticker",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "Return",
        "NextDayReturn",
        "Momentum_20D",
        "Volatility_20D",
        "AvgVolume_20D",
        "VolumeRatio_20D",
        "MarketReturn",
        "Momentum_20D_Z",
        "Volatility_20D_Z",
        "VolumeRatio_20D_Z",
    ]
].copy()


# STEP 10: Saving all output files
market_cap_df.to_csv("sp500_market_caps_ranked.csv", index=False)
pd.DataFrame({"Ticker": top50}).to_csv("top50_universe.csv", index=False)
long_data.to_csv("top50_stock_data_long.csv", index=False)
final_regression_dataset.to_csv("final_regression_dataset.csv", index=False)

print(f"\nSaved files successfully:")
print(f"- sp500_market_caps_ranked.csv")
print(f"- top50_universe.csv")
print(f"- top50_stock_data_long.csv")
print(f"- final_regression_dataset.csv")

print("\nFinal regression dataset shape:", final_regression_dataset.shape)
print(f"\nColumns in final regression dataset:")
print(final_regression_dataset.columns.tolist())

print(f"\nSample rows:")
print(final_regression_dataset.head())

/tmp/ipykernel_242/3915735090.py:21: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500_table = pd.read_html(response.text)[0]


Total tickers fetched: 503
  1. MMM: 78907564032
  2. AOS: 9215640576
  3. ABT: 188382216192
  4. ABBV: 399155298304
  5. ACN: 121677455360
  6. ADBE: 110945796096
  7. AMD: 321843101696
  8. AES: 10143275008
  9. AFL: 56718016512
 10. A: 31570563072
 11. APD: 64405475328
 12. ABNB: 76366086144
 13. AKAM: 15458837504
 14. ALB: 19348744192
 15. ARE: 8744736768
 16. ALGN: 12135077888
 17. ALLE: 12557427712
 18. LNT: 18406141952
 19. ALL: 53287899136
 20. GOOGL: 3678939643904
 21. GOOG: 3675552481280
 22. MO: 113562140672
 23. AMZN: 2253259800576
 24. AMCR: 18581168128
 25. AEE: 30423281664
 26. AEP: 71744462848
 27. AXP: 207902310400
 28. AIG: 41378467840
 29. AMT: 84285112320
 30. AWK: 27024689152
 31. AMP: 40969560064
 32. AME: 49629650944
 33. AMGN: 198942916608
 34. APH: 161143930880
 35. ADI: 149868920832
 36. AON: 68097937408
 37. APA: 11920471040
 38. APO: 58300547072
 39. AAPL: 3754291101696
 40. AMAT: 266633068544
 41. APP: 151929356288
 42. APTV: 15060733952
 43. ACGL: 34210131

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GEV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2018-01-01 -> 2024-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1514782800, endDate = 1704085200")')
/tmp/ipykernel_242/3915735090.py:101: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long_data = raw_data.stack(level=1).reset_index()



Saved files successfully:
- sp500_market_caps_ranked.csv
- top50_universe.csv
- top50_stock_data_long.csv
- final_regression_dataset.csv

Final regression dataset shape: (72221, 18)

Columns in final regression dataset:
['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Return', 'NextDayReturn', 'Momentum_20D', 'Volatility_20D', 'AvgVolume_20D', 'VolumeRatio_20D', 'MarketReturn', 'Momentum_20D_Z', 'Volatility_20D_Z', 'VolumeRatio_20D_Z']

Sample rows:
        Date Ticker       Open       High        Low      Close  Adj Close  \
0 2018-01-31   AAPL  41.717499  42.110001  41.625000  41.857498  39.174088   
1 2018-02-01   AAPL  41.792500  42.154999  41.689999  41.945000  39.255978   
2 2018-02-02   AAPL  41.500000  41.700001  40.025002  40.125000  37.552650   
3 2018-02-05   AAPL  39.775002  40.970001  39.000000  39.122501  36.614426   
4 2018-02-06   AAPL  38.707500  40.930000  38.500000  40.757500  38.144604   

        Volume    Return  NextDayReturn  Momentum_